# Sculpture Playground

Use this notebook to inspect the reconstructed sculpture model, rotate it, deform it, and export edited versions.

**Suggested workflow:**
1. Load the mesh in Cell 2.
2. Use the interactive viewer and rotation controls.
3. Try scale, translate, mirror, shear, and vertex-pull tools.
4. Compare the original and modified models.
5. Export the result when you are done.

In [ ]:
# Section 1: Load and inspect the sculpture model
from pathlib import Path

import numpy as np
import open3d as o3d
from IPython.display import display

from sculpture.playground import (
    ControlPoint,
    compute_bounding_box_info,
    deform_near_point,
    export_viewer_html,
    load_mesh,
    mesh_preview_figure,
    mirror_mesh,
    save_mesh,
    scale_mesh,
    shear_mesh,
    transform_vertex_subset,
    translate_mesh,
    rotate_mesh,
)

model_candidates = [
    Path("../data/output/meshes/mesh.ply"),
    Path("../data/output/reconstruction/point_cloud.ply"),
    Path("../data/output/wireframes/wireframe.obj"),
]

mesh = None
model_path = None
for candidate in model_candidates:
    if candidate.exists() and candidate.suffix.lower() in {".ply", ".obj", ".stl", ".off", ".glb", ".gltf"}:
        try:
            mesh = load_mesh(candidate)
            model_path = candidate
            break
        except Exception:
            continue

if mesh is None:
    model_path = Path("demo-sphere")
    mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
    mesh.compute_vertex_normals()
    print("No saved sculpture mesh was found, so a demo sphere is being used.")
else:
    print(f"Loaded: {model_path}")

stats = compute_bounding_box_info(mesh)
print({
    "vertices": len(mesh.vertices),
    "triangles": len(mesh.triangles),
    **stats,
})

mesh_preview_figure(mesh, title="Sculpture playground — original model").show()

## Section 2: Set Up an Interactive 3D Viewer

Use the Plotly viewer below to drag, orbit, and zoom around the sculpture directly inside the notebook.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

viewer_output = widgets.Output()
rotation_degrees = 0.0


def render_rotated_view(angle: float = 0.0) -> None:
    with viewer_output:
        clear_output(wait=True)
        rotated = rotate_mesh(mesh, angle)
        mesh_preview_figure(rotated, title=f"Interactive viewer — rotation {angle:.1f}°").show()


def _step(delta: float) -> None:
    global rotation_degrees
    rotation_degrees += delta
    render_rotated_view(rotation_degrees)


def _reset(_button) -> None:
    global rotation_degrees
    rotation_degrees = 0.0
    render_rotated_view(rotation_degrees)


def _export(_button) -> None:
    export_path = Path("../data/output/renders/sculpture_playground_export.html")
    exported = export_viewer_html(mesh, export_path, title="Sculpture playground viewer")
    with viewer_output:
        print(f"Exported HTML viewer to {exported}")

left_button = widgets.Button(description="⟲ Rotate -15°", button_style="info")
right_button = widgets.Button(description="Rotate +15° ⟳", button_style="info")
reset_button = widgets.Button(description="Reset", button_style="warning")
export_button = widgets.Button(description="Export HTML", button_style="success")

left_button.on_click(lambda button: _step(-15.0))
right_button.on_click(lambda button: _step(15.0))
reset_button.on_click(_reset)
export_button.on_click(_export)

toolbar = widgets.HBox([left_button, right_button, reset_button, export_button])
display(toolbar, viewer_output)
render_rotated_view(rotation_degrees)

## Section 3: Rotate the Model to View All Sides

Use the sliders below to inspect the sculpture from multiple angles. This is useful for studying silhouettes, grooves, and features that only appear from specific viewpoints.

In [ ]:
rotation_controls = widgets.Output()


def show_rotation(yaw=0.0, pitch=0.0, roll=0.0):
    with rotation_controls:
        clear_output(wait=True)
        rotated = mesh.clone()
        rotation = rotated.get_rotation_matrix_from_xyz(np.deg2rad([pitch, yaw, roll]))
        rotated.rotate(rotation, center=rotated.get_center())
        mesh_preview_figure(rotated, title=f"Rotation controls — yaw={yaw:.1f}, pitch={pitch:.1f}, roll={roll:.1f}").show()

rotation_ui = widgets.interactive(
    show_rotation,
    yaw=widgets.FloatSlider(min=0, max=360, step=5, value=0, description="Yaw"),
    pitch=widgets.FloatSlider(min=-180, max=180, step=5, value=0, description="Pitch"),
    roll=widgets.FloatSlider(min=-180, max=180, step=5, value=0, description="Roll"),
)
display(rotation_ui, rotation_controls)
show_rotation()

## Section 4: Translate and Scale the Model

Move the sculpture around in 3D space or resize it to see how positioning and scale affect the view.

In [ ]:
translate_scale_controls = widgets.Output()


def show_translate_scale(dx=0.0, dy=0.0, dz=0.0, scale=1.0):
    with translate_scale_controls:
        clear_output(wait=True)
        moved = translate_mesh(mesh, (dx, dy, dz))
        resized = scale_mesh(moved, scale)
        mesh_preview_figure(
            resized,
            title=f"Translate + scale — offset=({dx:.2f}, {dy:.2f}, {dz:.2f}), scale={scale:.2f}",
        ).show()

translate_scale_ui = widgets.interactive(
    show_translate_scale,
    dx=widgets.FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0, description="X"),
    dy=widgets.FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0, description="Y"),
    dz=widgets.FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0, description="Z"),
    scale=widgets.FloatSlider(min=0.2, max=2.5, step=0.05, value=1.0, description="Scale"),
)
display(translate_scale_ui, translate_scale_controls)
show_translate_scale()

## Section 5: Apply Basic Geometry Manipulations

Try simple edits like mirroring, shearing, pulling vertices near a control point, or moving only a subset of the mesh.

In [ ]:
# Create a few example variants for learning.
mirrored = mirror_mesh(mesh, axis="x")
sheared = shear_mesh(mesh, shear_xy=0.2, shear_xz=0.1, shear_yz=0.05)
control_point = ControlPoint(position=np.asarray(mesh.get_center()), radius=0.4, strength=0.3)
deformed = deform_near_point(mesh, control_point, direction=np.array([0.0, 0.0, 1.0]))
front_half = transform_vertex_subset(
    mesh,
    predicate=lambda vertex: vertex[2] >= np.asarray(mesh.get_center())[2],
    offset=(0.0, 0.0, 0.25),
)

print("Created mirrored, sheared, deformed, and subset-transformed variants.")
mesh_preview_figure(mirrored, title="Mirrored mesh").show()
mesh_preview_figure(sheared, title="Sheared mesh").show()
mesh_preview_figure(deformed, title="Pulled control point deformation").show()
mesh_preview_figure(front_half, title="Subset transformation").show()

## Section 6: Adjust Camera, Lighting, and Materials

Change the camera angle, zoom, shading, and color to make surface details easier to study.

In [ ]:
import plotly.graph_objects as go

camera_output = widgets.Output()


def show_camera_and_lighting(eye_x=1.8, eye_y=1.3, eye_z=1.1, opacity=0.9, color="#c6d8ff"):
    with camera_output:
        clear_output(wait=True)
        vertices = np.asarray(mesh.vertices)
        faces = np.asarray(mesh.triangles)
        figure = go.Figure(
            data=[
                go.Mesh3d(
                    x=vertices[:, 0],
                    y=vertices[:, 1],
                    z=vertices[:, 2],
                    i=faces[:, 0],
                    j=faces[:, 1],
                    k=faces[:, 2],
                    color=color,
                    opacity=opacity,
                    lighting=dict(ambient=0.35, diffuse=0.9, specular=0.35, roughness=0.75, fresnel=0.15),
                    lightposition=dict(x=120, y=50, z=80),
                )
            ]
        )
        figure.update_layout(
            title="Camera, lighting, and material controls",
            scene=dict(aspectmode="data", camera=dict(eye=dict(x=eye_x, y=eye_y, z=eye_z))),
        )
        figure.show()

camera_ui = widgets.interactive(
    show_camera_and_lighting,
    eye_x=widgets.FloatSlider(min=-4.0, max=4.0, step=0.1, value=1.8, description="Eye X"),
    eye_y=widgets.FloatSlider(min=-4.0, max=4.0, step=0.1, value=1.3, description="Eye Y"),
    eye_z=widgets.FloatSlider(min=-4.0, max=4.0, step=0.1, value=1.1, description="Eye Z"),
    opacity=widgets.FloatSlider(min=0.2, max=1.0, step=0.05, value=0.9, description="Opacity"),
    color=widgets.ColorPicker(value="#c6d8ff", description="Color"),
)
display(camera_ui, camera_output)
show_camera_and_lighting()

## Section 7: Compare Original vs. Modified Views

Display the original sculpture alongside one of the modified versions so you can inspect the differences side by side.

In [ ]:
from plotly.subplots import make_subplots

comparison = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Original", "Modified"),
)

for column, current_mesh, shade in [(1, mesh, "#9ecae1"), (2, deformed, "#f3b391")]:
    vertices = np.asarray(current_mesh.vertices)
    faces = np.asarray(current_mesh.triangles)
    comparison.add_trace(
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=shade,
            opacity=0.9,
            showscale=False,
        ),
        row=1,
        col=column,
    )

comparison.update_layout(
    title="Compare original vs. modified sculpture",
    scene=dict(aspectmode="data"),
    scene2=dict(aspectmode="data"),
)
comparison.show()

## Section 8: Save or Export the Updated Model

Export the edited sculpture to a file so you can reuse it in other tools, notebooks, or 3D applications.

In [ ]:
export_path = Path("../data/output/renders/sculpture_playground_export.html")
mesh_path = Path("../data/output/renders/sculpture_playground_export.ply")
exported_html = export_viewer_html(deformed, export_path, title="Modified sculpture viewer")
save_mesh(deformed, mesh_path)
print(f"Exported HTML viewer: {exported_html}")
print(f"Exported mesh: {mesh_path}")